# Week 2: Statistical Models for Time Series Forecasting

## Objective
Train and evaluate three statistical forecasting models on the cleaned time series data.

- **Training period:** 2013-01-01 to 2013-12-31
- **Test period:** 2014-01-01 to 2014-03-31

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load and Prepare Data

In [ ]:
RAW_PATH = "data/timeseries.csv"
CLEANED_PATH = "data/timeseries_cleaned.csv"

if not os.path.exists(CLEANED_PATH):
    print(f"Creating {CLEANED_PATH} from {RAW_PATH}...")
    df = pd.read_csv(RAW_PATH, parse_dates=["date"])
    df = df.sort_values("date").set_index("date").asfreq("D")
    df["unit_sales"] = df["unit_sales"].ffill().bfill()
    df.reset_index().to_csv(CLEANED_PATH, index=False)
else:
    print(f"Using existing {CLEANED_PATH}.")

df = pd.read_csv(CLEANED_PATH, parse_dates=["date"])
df = df.sort_values("date").set_index("date").asfreq("D")
print(df.head())
print(f"Shape: {df.shape}, Range: {df.index.min()} to {df.index.max()}")

## 2. Train / Test Split

In [ ]:
train = df.loc["2013-01-01":"2013-12-31"]
test = df.loc["2014-01-01":"2014-03-31"]

plt.plot(train.index, train["unit_sales"], label="Train")
plt.plot(test.index, test["unit_sales"], label="Test", color="orange")
plt.title("Train / Test Split")
plt.legend()
plt.show()

## 3. Evaluation Utilities

In [ ]:
def evaluate(true, pred, model_name):
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true, pred)
    r2 = r2_score(true, pred)
    print(f"{model_name}  ->  MSE: {mse:.2f} | RMSE: {rmse:.2f} | MAE: {mae:.2f} | R²: {r2:.4f}")
    return {"Model": model_name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

results = []

def plot_forecast(train, test, pred, title="Forecast"):
    plt.plot(train.index, train["unit_sales"], label="Train")
    plt.plot(test.index, test["unit_sales"], label="Actual", color="orange")
    plt.plot(test.index, pred, label="Forecast", color="red", linestyle="--")
    plt.title(title)
    plt.legend()
    plt.show()

## 4. Model 1: ARIMA

In [ ]:
from statsmodels.tsa.stattools import adfuller
adf_result = adfuller(train["unit_sales"].dropna())
print(f"ADF Statistic: {adf_result[0]:.4f}, p-value: {adf_result[1]:.4f}")
d = 1

In [ ]:
best_aic = np.inf
best_order = None
best_model = None

for p in range(0, 4):
    for q in range(0, 4):
        try:
            model = ARIMA(train["unit_sales"], order=(p, d, q))
            fitted = model.fit()
            if fitted.aic < best_aic:
                best_aic = fitted.aic
                best_order = (p, d, q)
                best_model = fitted
        except Exception:
            continue

print(f"Best ARIMA order: {best_order} (AIC: {best_aic:.2f})")
forecast_arima = best_model.forecast(steps=len(test))
plot_forecast(train, test, forecast_arima, title="ARIMA Forecast")
results.append(evaluate(test["unit_sales"], forecast_arima, "ARIMA"))

## 5. Model 2: SARIMA

In [ ]:
order = best_order if best_order else (2, 1, 2)
seasonal_order = (1, 1, 1, 7)

sarima_model = SARIMAX(
    train["unit_sales"],
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_result = sarima_model.fit(disp=False)
print(sarima_result.summary())

forecast_sarima = sarima_result.get_forecast(steps=len(test)).predicted_mean
plot_forecast(train, test, forecast_sarima, title="SARIMA Forecast")
results.append(evaluate(test["unit_sales"], forecast_sarima, "SARIMA"))

## 6. Model 3: Exponential Smoothing (Holt-Winters)

In [ ]:
hw_model = ExponentialSmoothing(
    train["unit_sales"],
    trend="add",
    seasonal="add",
    seasonal_periods=7
).fit()

forecast_hw = hw_model.forecast(steps=len(test))
plot_forecast(train, test, forecast_hw, title="Holt-Winters Forecast")
results.append(evaluate(test["unit_sales"], forecast_hw, "Holt-Winters"))

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, x="Model", y="RMSE")
plt.title("Model Comparison (RMSE)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()